# ORIENTA · Entrenamiento del recomendador por contenido
Modelo TF-IDF real sobre síntesis trazables de programas institucionales. No requiere inventar estudiantes ni etiquetas de afinidad. Sus resultados expresan coincidencia temática; no son diagnósticos ni probabilidades de éxito.


In [ ]:
!git clone --branch feature/completar-catalogo-recomendador https://github.com/HuamaniAlexander/Orienta-MVP1.git orienta
%cd orienta
!pip -q install uv
!uv export --frozen --no-dev --extra ml --no-hashes --format requirements-txt -o /tmp/orienta-content-requirements.txt
!pip -q install -r /tmp/orienta-content-requirements.txt

## Revisar el corpus
Los textos son síntesis académicas breves con fuentes. Revisar cada síntesis y documento antes de actualizar. Una fuente nueva no implica que el contenido se aplique a todas las sedes de una institución.


In [ ]:
import json
from pathlib import Path

corpus = json.loads(Path("data/ml/corpus_carreras.json").read_text())
[(r["career_id"], r["slug"], r["source_url"]) for r in corpus]

## Ajustar y exportar
Se aprenden vocabulario e IDF; no se ejecuta entrenamiento dentro de la API. El artefacto JSON evita deserializar pickle. No se informa accuracy/F1 porque no existen etiquetas supervisadas para calcularlas.


In [ ]:
from ml.src.train_content import train

report = train("data/ml/corpus_carreras.json", "data/models/contenido.json")
report

## Verificar inferencia
Estas consultas son pruebas de software sobre temas explícitos, no encuestas de alumnos ni validación psicométrica.


In [ ]:
import sys

sys.path.insert(0, "backend")
from app.adapters.outbound.ml.content_engine import ContentEngine

engine = ContentEngine("data/models/contenido.json")
responses = {f"Q{i:02}": 1 for i in range(1, 9)}
responses["Q03"] = 5
[(r["slug"], engine.evaluate(responses, r["career_id"])["score"]) for r in corpus]

## Entregar al backend
Descargar ambos archivos y sustituirlos juntos en `data/models/`. Ejecutar pruebas antes de publicar. `CONTENT_MODEL_PATH` apunta al JSON y FastAPI carga el modelo una vez al arrancar. No hace falta mantener Colab abierto. El aprendizaje supervisado opcional conserva su notebook separado y requiere datos reales etiquetados.


In [ ]:
from google.colab import files

files.download("data/models/contenido.json")
files.download("data/models/contenido.manifest.json")